In [ ]:
import numpy as np
import scipy.signal as signal
import scanpy as sc
from scipy.sparse import csr_matrix
from multiprocessing import Pool, cpu_count

In [ ]:
def tic_normalization(adata):
    """Normalize spectra based on Total Ion Current (TIC) for sparse matrices."""
    if isinstance(adata.X, csr_matrix):
        tic = np.array(adata.X.sum(axis=1)).flatten()
    else:
        tic = np.sum(adata.X, axis=1, keepdims=True)
    
    tic[tic == 0] = 1  # Avoid division by zero
    adata.layers["TIC_normalized"] = adata.X / tic[:, None]
    return adata

def baseline_correction(spectra, lambda_=1e4, p=0.01, niter=10):
    """Asymmetric Least Squares (ALS) baseline correction."""
    L = len(spectra)
    D = np.diff(np.eye(L), 2)
    w = np.ones(L)
    for _ in range(niter):
        W = np.diag(w)
        Z = np.linalg.inv(W + lambda_ * D.T @ D) @ (W @ spectra)
        w = p * (spectra > Z) + (1 - p) * (spectra < Z)
    return spectra - Z

def smooth_spectra(spectra, window=7, polyorder=3):
    """Apply Savitzky-Golay filter for smoothing."""
    return signal.savgol_filter(spectra, window, polyorder)

def detect_peaks(spectra, prominence=0.01):
    """Detect peaks using prominence thresholding."""
    peaks, properties = signal.find_peaks(spectra, prominence=prominence)
    return peaks, properties["prominences"]

def process_single_spectrum(spectrum, prominence=0.01):
    """Process a single spectrum: baseline correction, smoothing, and peak detection."""
    spectrum = baseline_correction(spectrum)
    spectrum = smooth_spectra(spectrum)
    peaks, _ = detect_peaks(spectrum, prominence=prominence)
    processed_spectrum = np.zeros_like(spectrum)
    processed_spectrum[peaks] = spectrum[peaks]
    return processed_spectrum

def process_maldi_data(adata, bin_size=0.1, prominence=0.01, top_n=100, output_file="processed_data.h5ad"):
    """Parallel processing for MALDI data."""
    adata = tic_normalization(adata)
    spectra_list = [adata.layers["TIC_normalized"][i] for i in range(adata.X.shape[0])]
    
    with Pool(cpu_count()) as pool:
        processed_data = pool.starmap(process_single_spectrum, [(s, prominence) for s in spectra_list])
    
    adata.layers["processed"] = np.array(processed_data)
    selected_peaks = select_top_n_peaks(adata, n=top_n)
    adata.uns["selected_peaks"] = selected_peaks  # Store selected peaks
    binned_mz = adaptive_binning(selected_peaks, bin_size)
    adata.uns["binned_mz"] = binned_mz
    adata.write(output_file)
    print(f"Processed data saved to: {output_file}")
    return adata

In [ ]:
# Define file paths
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad"
output_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_processed.h5ad"

# Load and process AnnData
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")
adata = process_maldi_data(adata, top_n=50, output_file=output_file)
print(f"Processed data saved to: {output_file}")
print("Selected peaks (m/z values):", adata.uns["selected_peaks"])

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_processed.h5ad")
print("Reloaded processed AnnData object.")
print("Stored layers:", adata.layers.keys())  # Check available layers
print("Selected peaks:", adata.uns["selected_peaks"])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
from sklearn.decomposition import PCA

# Load processed AnnData
adata = sc.read_h5ad("/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_processed.h5ad")

# Extract top N peak intensities
selected_peaks = adata.uns["selected_peaks"]  # Top peaks (m/z values)
processed_spectra = adata.layers["processed"]  # Processed spectra

# Get only selected peaks
selected_spectra = processed_spectra[:, selected_peaks]

# Perform PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(selected_spectra)

# Extract sample groups (assuming 'group' column exists in `adata.obs`)
if "group" in adata.obs.columns:
    sample_groups = adata.obs["group"]
else:
    sample_groups = np.array(["Unknown"] * adata.shape[0])  # Default if no group info

# Unique groups and colors
unique_groups = np.unique(sample_groups)
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_groups)))

# Plot PCA with group colors
plt.figure(figsize=(8, 6))
for i, group in enumerate(unique_groups):
    mask = sample_groups == group
    plt.scatter(pca_result[mask, 0], pca_result[mask, 1], label=group, color=colors[i], alpha=0.7, edgecolors="k")

# Customize highlight samples (e.g., outliers or samples of interest)
highlight_samples = [0, 5, 10]  # Specify indices of samples to highlight (adjust as needed)
highlight_color = 'green'  # Custom highlight color
highlight_marker = 'X'  # Custom highlight marker (e.g., 'X', 's' for square, 'o' for circle)

# Plot highlighted samples with custom color and marker
for idx in highlight_samples:
    plt.scatter(pca_result[idx, 0], pca_result[idx, 1], color=highlight_color, 
                marker=highlight_marker, s=120, label=f"Highlighted {idx}", edgecolors='k')

# Label the highlighted samples
for idx in highlight_samples:
    plt.text(pca_result[idx, 0], pca_result[idx, 1], f"Sample {idx}", fontsize=9, ha='right', color='black')

plt.xlabel("PC1 (%.2f%% Variance)" % (pca.explained_variance_ratio_[0] * 100))
plt.ylabel("PC2 (%.2f%% Variance)" % (pca.explained_variance_ratio_[1] * 100))
plt.title("PCA of Top Peak Intensities (Colored by Group with Custom Highlight)")
plt.legend(title="Sample Group")
plt.grid()
plt.show()


In [ ]:
plot_mz_image(adata,760.5856)

In [ ]:
plot_normalized_mz_image(adata, mz_value=760.5856)

In [ ]:
plot_mz_image(adata, mz_value=798.5417)

In [ ]:
plot_normalized_mz_image(adata, mz_value=798.5417)